# 03 - Feature Engineering (Leakage-Safe)

Goal: build lag, rolling, calendar, and business features for demand forecasting
while guaranteeing no future information leaks into any feature.

**Leakage rule enforced throughout:** a feature for day T may only use
information available strictly before day T's `units_sold` is known.
See `src/feature_engineering.py` docstring for the full reasoning per feature type.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src.feature_engineering import build_features, FEATURE_COLUMNS, CATEGORICAL_FEATURE_COLUMNS
from src.utils import PROCESSED_DATA_PATH

df = pd.read_csv(PROCESSED_DATA_PATH, parse_dates=['date'])
df.shape

(73100, 15)

## Feature groups

- **Lag features**: `lag_1, lag_7, lag_14, lag_28` -- yesterday's / last week's / etc. actual demand for that store-product.
- **Rolling features**: `rolling_mean_{7,14,28}`, `rolling_std_{7,28}` -- computed on a window that is `shift(1)`-ed first, so the window for day T never includes day T itself.
- **Calendar features**: day_of_week, week_of_year, month, quarter, year, is_weekend -- known in advance, not derived from sales, so safe to use as-of day T.
- **Business features**: promotion, price, inventory_level, discount_pct, price_vs_competitor(_pct) -- known ahead of day T in a real deployment (prices/promo calendars are set in advance; inventory_level is the day's opening stock, a system record).
- **Leak-safe target encoding**: `cat_expanding_mean` -- an expanding mean of demand per category computed only over each row's own past (`shift(1).expanding()`), never the full-dataset average.

In [2]:
feat_df = build_features(df)
print("Shape after feature engineering:", feat_df.shape)
print("Rows dropped for incomplete history (< 28 days into a series):", len(df) - len(feat_df))

Shape after feature engineering: (70300, 33)
Rows dropped for incomplete history (< 28 days into a series): 2800


In [3]:
feat_df[FEATURE_COLUMNS].isnull().sum().sum()

np.int64(0)

Zero NaNs left in any feature column -- confirms no leaked/garbage values slipped through.

In [4]:
feat_df[['date','store_id','product_id','units_sold'] + FEATURE_COLUMNS[:6]].head(10)

,date,store_id,product_id,units_sold,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14
0,2022-01-29,S001,P0001,47,97.0,191.0,176.0,127.0,111.142857,137.357143
1,2022-01-30,S001,P0001,130,47.0,79.0,97.0,81.0,90.571429,128.142857
2,2022-01-31,S001,P0001,200,130.0,9.0,69.0,5.0,97.857143,130.500000
3,2022-02-01,S001,P0001,279,200.0,189.0,123.0,58.0,125.142857,139.857143
4,2022-02-02,S001,P0001,38,279.0,49.0,211.0,147.0,138.000000,151.000000
5,2022-02-03,S001,P0001,71,38.0,164.0,79.0,37.0,136.428571,138.642857
6,2022-02-04,S001,P0001,25,71.0,97.0,390.0,107.0,123.142857,138.071429
7,2022-02-05,S001,P0001,350,25.0,47.0,191.0,2.0,112.857143,112.000000
8,2022-02-06,S001,P0001,69,350.0,130.0,79.0,350.0,156.142857,123.357143
9,2022-02-07,S001,P0001,62,69.0,200.0,9.0,36.0,147.428571,122.642857


## Sanity check: verifying no leakage manually

For a single store-product series, `lag_1` on day T should equal `units_sold`
on day T-1 exactly -- if this weren't true (e.g. off-by-zero shift), our
model would be cheating by seeing the current day's answer.

In [5]:
check = feat_df[(feat_df.store_id=='S001') & (feat_df.product_id=='P0001')].sort_values('date').head(5)
check[['date','units_sold','lag_1']]

,date,units_sold,lag_1
0,2022-01-29,47,97.0
1,2022-01-30,130,47.0
2,2022-01-31,200,130.0
3,2022-02-01,279,200.0
4,2022-02-02,38,279.0


In [6]:
from src.utils import FEATURES_DATA_PATH
feat_df.to_csv(FEATURES_DATA_PATH, index=False)
print("Saved:", feat_df.shape, "->", FEATURES_DATA_PATH)

Saved: (70300, 33) -> c:\Users\Pdllab\Downloads\demand-forecasting-inventory-optimization (2)\demand-forecasting-inventory-optimization\data\processed\features.csv
